# Tabular Learners on the 222 Engineered Features

## Purpose

The engineered-feature Random Forest achieved a mean subject-fold macro-F1 of
0.7257. This experiment keeps the examples, 222 deterministic features, subject
folds, and evaluation metrics fixed while changing only the learning algorithm.

Two challengers are evaluated:

1. standardized multinomial logistic regression, as a linear diagnostic;
2. histogram gradient boosting, as the main nonlinear tabular challenger.

This comparison asks whether the engineered representation is already strongly
linearly separable and whether sequentially boosted trees can use it more
effectively than the existing bagged Random Forest.

Each challenger receives one predeclared configuration. No hyperparameter
search is performed. Standardization, where required, is fitted only on the
outer-training subjects of each fold. The histogram model's automatic early
stopping is disabled because its default row-wise validation split would not
respect subject boundaries.

The primary metric remains mean macro-F1 across the five fixed subject folds.
Accuracy, balanced accuracy, fold standard deviation, and fold-level macro-F1
are also reported.

A challenger will replace the Random Forest only if it:

- reaches at least 0.7357 mean macro-F1, representing a gain of at least 0.01;
- outperforms the Random Forest on at least 3 of the 5 folds.

The ten reserved test subjects remain untouched.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            candidate
            / "modeling"
            / "artifacts"
            / "engineered_222_meta.json"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the activity_project root."
    )

MODEL_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modeling"
    / "artifacts"
)

FEATURE_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_train.parquet"
)

FEATURE_META_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_meta.json"
)

TRAIN_METADATA_PATH = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
    / "train_metadata.parquet"
)

DEVELOPMENT_FOLDS_PATH = (
    PROJECT_ROOT
    / "baseline"
    / "artifacts"
    / "development_folds.json"
)

feature_meta = json.loads(
    FEATURE_META_PATH.read_text(
        encoding="utf-8"
    )
)

development_folds = json.loads(
    DEVELOPMENT_FOLDS_PATH.read_text(
        encoding="utf-8"
    )
)

feature_table = pd.read_parquet(
    FEATURE_PATH
)

train_metadata = pd.read_parquet(
    TRAIN_METADATA_PATH
)

feature_names = feature_meta[
    "feature_names"
]

assert feature_table.columns.tolist() == [
    "array_index",
    *feature_names,
]

assert np.array_equal(
    feature_table[
        "array_index"
    ].to_numpy(),
    train_metadata[
        "array_index"
    ].to_numpy(),
)

X = feature_table[
    feature_names
].to_numpy()

y = (
    train_metadata[
        "activity_code"
    ]
    .astype(str)
    .to_numpy()
)

fold_assignments = (
    train_metadata[
        "fold"
    ]
    .astype(int)
    .to_numpy()
)

subject_ids = (
    train_metadata[
        "user_id"
    ].to_numpy()
)

expected_fold_by_subject = {
    int(subject_id): int(fold)
    for fold, subjects in (
        development_folds[
            "validation_subject_ids_by_fold"
        ].items()
    )
    for subject_id in subjects
}

observed_fold_by_subject = {
    int(row.user_id): int(row.fold)
    for row in (
        train_metadata[
            ["user_id", "fold"]
        ]
        .drop_duplicates()
        .itertuples(index=False)
    )
}

assert X.shape == (
    25_585,
    222,
)
assert np.isfinite(X).all()
assert observed_fold_by_subject == (
    expected_fold_by_subject
)

print(
    f"Examples: {len(y):,}"
)
print(
    f"Features: {X.shape[1]}"
)
print(
    f"Subjects: "
    f"{np.unique(subject_ids).size}"
)
print(
    f"Activities: "
    f"{np.unique(y).size}"
)
print(
    "Folds:",
    sorted(
        np.unique(
            fold_assignments
        )
    ),
)
print(
    "Feature table aligned with "
    "the authoritative metadata."
)

Examples: 25,585
Features: 222
Subjects: 41
Activities: 18
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Feature table aligned with the authoritative metadata.


## 2. Linear diagnostic: multinomial logistic regression

Logistic regression tests how much activity discrimination is available through
linear combinations of the 222 engineered features.

The fixed configuration uses:

- fold-specific standardization;
- multinomial classification through the `lbfgs` solver;
- L2 regularization with `C=1.0`;
- no class weighting;
- a maximum of 2,000 optimization iterations.

The scaler is fitted only on the outer-training examples in each fold. No inner
validation is required because the regularization strength and stopping
configuration are fixed in advance.

In [2]:
from time import perf_counter

from sklearn.linear_model import (
    LogisticRegression,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
)


RANDOM_STATE = 42

RF_MEAN_MACRO_F1 = 0.7257

RF_FOLD_MACRO_F1 = {
    1: 0.7673,
    2: 0.6634,
    3: 0.7675,
    4: 0.6992,
    5: 0.7310,
}

TABULAR_IMPROVEMENT_TARGET = 0.7357
MIN_FOLDS_IMPROVED = 3

ACTIVITY_CODES = np.sort(
    np.unique(y)
)


def calculate_metrics(
    y_true,
    y_pred,
):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
    }


logistic_prediction_parts = []
logistic_fold_rows = []

for outer_fold in sorted(
    np.unique(fold_assignments)
):
    training_mask = (
        fold_assignments
        != outer_fold
    )

    validation_mask = (
        fold_assignments
        == outer_fold
    )

    training_subjects = set(
        subject_ids[
            training_mask
        ]
    )

    validation_subjects = set(
        subject_ids[
            validation_mask
        ]
    )

    assert training_subjects.isdisjoint(
        validation_subjects
    )

    model = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=2_000,
                tol=1e-4,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    start_time = perf_counter()

    model.fit(
        X[training_mask],
        y[training_mask],
    )

    fit_seconds = (
        perf_counter()
        - start_time
    )

    y_pred = model.predict(
        X[validation_mask]
    )

    fold_metrics = calculate_metrics(
        y[validation_mask],
        y_pred,
    )

    optimizer_iterations = int(
        model.named_steps[
            "classifier"
        ].n_iter_.max()
    )

    logistic_fold_rows.append({
        "fold": int(outer_fold),
        **fold_metrics,
        "rf_macro_f1": (
            RF_FOLD_MACRO_F1[
                int(outer_fold)
            ]
        ),
        "fit_seconds": fit_seconds,
        "optimizer_iterations": (
            optimizer_iterations
        ),
    })

    logistic_prediction_parts.append(
        pd.DataFrame({
            "array_index": (
                train_metadata.loc[
                    validation_mask,
                    "array_index",
                ].to_numpy()
            ),
            "outer_fold": int(
                outer_fold
            ),
            "y_true": y[
                validation_mask
            ],
            "y_pred": y_pred,
        })
    )

    print(
        f"Fold {outer_fold}: "
        f"macro-F1="
        f"{fold_metrics['macro_f1']:.4f}, "
        f"iterations="
        f"{optimizer_iterations}, "
        f"time={fit_seconds:.1f}s"
    )

logistic_fold_metrics = (
    pd.DataFrame(
        logistic_fold_rows
    )
)

logistic_fold_metrics[
    "gain_vs_rf"
] = (
    logistic_fold_metrics[
        "macro_f1"
    ]
    - logistic_fold_metrics[
        "rf_macro_f1"
    ]
)

logistic_oof_predictions = (
    pd.concat(
        logistic_prediction_parts,
        ignore_index=True,
    )
    .sort_values("array_index")
    .reset_index(drop=True)
)

assert len(
    logistic_oof_predictions
) == len(train_metadata)

assert logistic_oof_predictions[
    "array_index"
].is_unique

logistic_macro_f1_mean = (
    logistic_fold_metrics[
        "macro_f1"
    ].mean()
)

logistic_folds_better_than_rf = int(
    logistic_fold_metrics[
        "gain_vs_rf"
    ].gt(0).sum()
)

logistic_summary = pd.Series({
    "accuracy_mean": (
        logistic_fold_metrics[
            "accuracy"
        ].mean()
    ),
    "balanced_accuracy_mean": (
        logistic_fold_metrics[
            "balanced_accuracy"
        ].mean()
    ),
    "macro_f1_mean": (
        logistic_macro_f1_mean
    ),
    "macro_f1_fold_std": (
        logistic_fold_metrics[
            "macro_f1"
        ].std()
    ),
    "gain_vs_rf_mean": (
        logistic_macro_f1_mean
        - RF_MEAN_MACRO_F1
    ),
    "folds_better_than_rf": (
        logistic_folds_better_than_rf
    ),
    "passes_improvement_gate": (
        logistic_macro_f1_mean
        >= TABULAR_IMPROVEMENT_TARGET
        and logistic_folds_better_than_rf
        >= MIN_FOLDS_IMPROVED
    ),
})

display(
    logistic_fold_metrics.round(4)
)

display(
    logistic_summary
    .to_frame("value")
    .round(4)
)

Fold 1: macro-F1=0.7555, iterations=438, time=1.5s
Fold 2: macro-F1=0.6647, iterations=465, time=1.6s
Fold 3: macro-F1=0.7643, iterations=451, time=1.5s
Fold 4: macro-F1=0.6692, iterations=448, time=1.5s
Fold 5: macro-F1=0.7091, iterations=425, time=1.4s


,fold,accuracy,balanced_accuracy,macro_f1,rf_macro_f1,fit_seconds,optimizer_iterations,gain_vs_rf
0,1,0.7589,0.7589,0.7555,0.7673,1.5320,438,-0.0118
1,2,0.6658,0.6664,0.6647,0.6634,1.6055,465,0.0013
2,3,0.7646,0.7657,0.7643,0.7675,1.4539,451,-0.0032
3,4,0.6698,0.6709,0.6692,0.6992,1.4871,448,-0.0300
4,5,0.7121,0.7137,0.7091,0.7310,1.3820,425,-0.0219


,value
accuracy_mean,0.714239
balanced_accuracy_mean,0.715115
macro_f1_mean,0.712557
macro_f1_fold_std,0.046648
gain_vs_rf_mean,-0.013143
folds_better_than_rf,1
passes_improvement_gate,False


### Linear-diagnostic result

The standardized logistic regression achieves mean macro-F1
**0.7126 ± 0.0466**. All five fits converge well before the 2,000-iteration
limit.

This is only 0.0131 below the engineered Random Forest, indicating that the
222-feature representation exposes substantial linearly separable activity
structure. Nevertheless, logistic regression improves on the RF only in fold 2,
by 0.0013, and performs worse overall. Its largest deficits occur in folds 4
and 5.

The linear model therefore provides a strong diagnostic benchmark but does not
pass the replacement gate. The next experiment tests whether gradient-boosted
trees can model the remaining nonlinear structure more effectively than the
Random Forest.

## 3. Histogram gradient boosting

Histogram gradient boosting builds trees sequentially, with each stage
attempting to correct errors made by the preceding stages. This differs from
the Random Forest, whose trees are independently fitted and averaged.

One fixed, moderately regularized configuration is used:

- learning rate: 0.05;
- boosting iterations: 200;
- maximum leaf nodes per tree: 31;
- minimum samples per leaf: 20;
- L2 leaf regularization: 1.0;
- no class weighting.

The features are not standardized because tree splits are invariant to feature
scale. Automatic early stopping is disabled to prevent a random row-wise
validation split. The iteration count is fixed in advance, so no outer-fold
information influences training duration.

In [3]:
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
)


HGB_CONFIG = {
    "learning_rate": 0.05,
    "max_iter": 200,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
    "early_stopping": False,
    "random_state": RANDOM_STATE,
}

hgb_prediction_parts = []
hgb_fold_rows = []

for outer_fold in sorted(
    np.unique(fold_assignments)
):
    training_mask = (
        fold_assignments
        != outer_fold
    )

    validation_mask = (
        fold_assignments
        == outer_fold
    )

    model = (
        HistGradientBoostingClassifier(
            **HGB_CONFIG
        )
    )

    start_time = perf_counter()

    model.fit(
        X[training_mask],
        y[training_mask],
    )

    fit_seconds = (
        perf_counter()
        - start_time
    )

    y_pred = model.predict(
        X[validation_mask]
    )

    fold_metrics = calculate_metrics(
        y[validation_mask],
        y_pred,
    )

    hgb_fold_rows.append({
        "fold": int(outer_fold),
        **fold_metrics,
        "rf_macro_f1": (
            RF_FOLD_MACRO_F1[
                int(outer_fold)
            ]
        ),
        "fit_seconds": fit_seconds,
        "iterations": int(
            model.n_iter_
        ),
    })

    hgb_prediction_parts.append(
        pd.DataFrame({
            "array_index": (
                train_metadata.loc[
                    validation_mask,
                    "array_index",
                ].to_numpy()
            ),
            "outer_fold": int(
                outer_fold
            ),
            "y_true": y[
                validation_mask
            ],
            "y_pred": y_pred,
        })
    )

    print(
        f"Fold {outer_fold}: "
        f"macro-F1="
        f"{fold_metrics['macro_f1']:.4f}, "
        f"iterations={model.n_iter_}, "
        f"time={fit_seconds:.1f}s"
    )

hgb_fold_metrics = pd.DataFrame(
    hgb_fold_rows
)

hgb_fold_metrics[
    "gain_vs_rf"
] = (
    hgb_fold_metrics[
        "macro_f1"
    ]
    - hgb_fold_metrics[
        "rf_macro_f1"
    ]
)

hgb_oof_predictions = (
    pd.concat(
        hgb_prediction_parts,
        ignore_index=True,
    )
    .sort_values("array_index")
    .reset_index(drop=True)
)

assert len(
    hgb_oof_predictions
) == len(train_metadata)

assert hgb_oof_predictions[
    "array_index"
].is_unique

hgb_macro_f1_mean = (
    hgb_fold_metrics[
        "macro_f1"
    ].mean()
)

hgb_folds_better_than_rf = int(
    hgb_fold_metrics[
        "gain_vs_rf"
    ].gt(0).sum()
)

hgb_summary = pd.Series({
    "accuracy_mean": (
        hgb_fold_metrics[
            "accuracy"
        ].mean()
    ),
    "balanced_accuracy_mean": (
        hgb_fold_metrics[
            "balanced_accuracy"
        ].mean()
    ),
    "macro_f1_mean": (
        hgb_macro_f1_mean
    ),
    "macro_f1_fold_std": (
        hgb_fold_metrics[
            "macro_f1"
        ].std()
    ),
    "gain_vs_rf_mean": (
        hgb_macro_f1_mean
        - RF_MEAN_MACRO_F1
    ),
    "folds_better_than_rf": (
        hgb_folds_better_than_rf
    ),
    "passes_improvement_gate": (
        hgb_macro_f1_mean
        >= TABULAR_IMPROVEMENT_TARGET
        and hgb_folds_better_than_rf
        >= MIN_FOLDS_IMPROVED
    ),
})

display(
    hgb_fold_metrics.round(4)
)

display(
    hgb_summary
    .to_frame("value")
    .round(4)
)

Fold 1: macro-F1=0.7852, iterations=200, time=57.5s
Fold 2: macro-F1=0.6895, iterations=200, time=52.6s
Fold 3: macro-F1=0.7887, iterations=200, time=55.0s
Fold 4: macro-F1=0.7159, iterations=200, time=52.6s
Fold 5: macro-F1=0.7434, iterations=200, time=51.6s


,fold,accuracy,balanced_accuracy,macro_f1,rf_macro_f1,fit_seconds,iterations,gain_vs_rf
0,1,0.7868,0.7868,0.7852,0.7673,57.4872,200,0.0179
1,2,0.6861,0.6874,0.6895,0.6634,52.6387,200,0.0261
2,3,0.7885,0.7897,0.7887,0.7675,55.0072,200,0.0212
3,4,0.7066,0.7072,0.7159,0.6992,52.6001,200,0.0167
4,5,0.7409,0.7425,0.7434,0.7310,51.5891,200,0.0124


,value
accuracy_mean,0.741784
balanced_accuracy_mean,0.742695
macro_f1_mean,0.744552
macro_f1_fold_std,0.043192
gain_vs_rf_mean,0.018852
folds_better_than_rf,5
passes_improvement_gate,True


In [4]:
tabular_fold_comparison = pd.DataFrame({
    "random_forest": pd.Series(
        RF_FOLD_MACRO_F1
    ),
    "logistic_regression": (
        logistic_fold_metrics
        .set_index("fold")[
            "macro_f1"
        ]
    ),
    "histogram_gradient_boosting": (
        hgb_fold_metrics
        .set_index("fold")[
            "macro_f1"
        ]
    ),
})

tabular_fold_comparison.index.name = (
    "fold"
)

tabular_fold_comparison[
    "logistic_minus_rf"
] = (
    tabular_fold_comparison[
        "logistic_regression"
    ]
    - tabular_fold_comparison[
        "random_forest"
    ]
)

tabular_fold_comparison[
    "boosting_minus_rf"
] = (
    tabular_fold_comparison[
        "histogram_gradient_boosting"
    ]
    - tabular_fold_comparison[
        "random_forest"
    ]
)

tabular_summary = pd.DataFrame([
    {
        "model": (
            "Logistic regression"
        ),
        "accuracy_mean": (
            logistic_summary[
                "accuracy_mean"
            ]
        ),
        "balanced_accuracy_mean": (
            logistic_summary[
                "balanced_accuracy_mean"
            ]
        ),
        "macro_f1_mean": (
            logistic_summary[
                "macro_f1_mean"
            ]
        ),
        "macro_f1_fold_std": (
            logistic_summary[
                "macro_f1_fold_std"
            ]
        ),
        "gain_vs_rf": (
            logistic_summary[
                "gain_vs_rf_mean"
            ]
        ),
        "folds_better_than_rf": (
            logistic_summary[
                "folds_better_than_rf"
            ]
        ),
        "passes_replacement_gate": (
            logistic_summary[
                "passes_improvement_gate"
            ]
        ),
    },
    {
        "model": (
            "Engineered Random Forest"
        ),
        "accuracy_mean": 0.7261,
        "balanced_accuracy_mean": 0.7270,
        "macro_f1_mean": (
            RF_MEAN_MACRO_F1
        ),
        "macro_f1_fold_std": 0.0449,
        "gain_vs_rf": 0.0,
        "folds_better_than_rf": 0,
        "passes_replacement_gate": False,
    },
    {
        "model": (
            "Histogram gradient boosting"
        ),
        "accuracy_mean": (
            hgb_summary[
                "accuracy_mean"
            ]
        ),
        "balanced_accuracy_mean": (
            hgb_summary[
                "balanced_accuracy_mean"
            ]
        ),
        "macro_f1_mean": (
            hgb_summary[
                "macro_f1_mean"
            ]
        ),
        "macro_f1_fold_std": (
            hgb_summary[
                "macro_f1_fold_std"
            ]
        ),
        "gain_vs_rf": (
            hgb_summary[
                "gain_vs_rf_mean"
            ]
        ),
        "folds_better_than_rf": (
            hgb_summary[
                "folds_better_than_rf"
            ]
        ),
        "passes_replacement_gate": (
            hgb_summary[
                "passes_improvement_gate"
            ]
        ),
    },
])

display(
    tabular_fold_comparison.round(4)
)

display(
    tabular_summary
    .sort_values(
        "macro_f1_mean",
        ascending=False,
    )
    .reset_index(drop=True)
    .round(4)
)

,random_forest,logistic_regression,histogram_gradient_boosting,logistic_minus_rf,boosting_minus_rf
fold,,,,,
1,0.7673,0.7555,0.7852,-0.0118,0.0179
2,0.6634,0.6647,0.6895,0.0013,0.0261
3,0.7675,0.7643,0.7887,-0.0032,0.0212
4,0.6992,0.6692,0.7159,-0.0300,0.0167
5,0.7310,0.7091,0.7434,-0.0219,0.0124


,model,accuracy_mean,balanced_accuracy_mean,macro_f1_mean,macro_f1_fold_std,gain_vs_rf,folds_better_than_rf,passes_replacement_gate
0,Histogram gradient boosting,0.7418,0.7427,0.7446,0.0432,0.0189,5,True
1,Engineered Random Forest,0.7261,0.7270,0.7257,0.0449,0.0000,0,False
2,Logistic regression,0.7142,0.7151,0.7126,0.0466,-0.0131,1,False


## 4. Findings and decision

The learner comparison separates the contribution of the engineered
representation from that of the classification algorithm.

Standardized multinomial logistic regression achieves mean macro-F1
**0.7126 ± 0.0466**. Its strong performance shows that the 222 engineered
features expose substantial linearly separable activity structure. However, it
remains 0.0131 below the Random Forest and exceeds it only on fold 2, so it does
not pass the replacement gate.

Histogram gradient boosting achieves mean macro-F1 **0.7446 ± 0.0432**, with
mean accuracy 0.7418 and mean balanced accuracy 0.7427. It improves on the
engineered Random Forest by **0.0189 macro-F1** and performs better on all five
subject folds. The improvement is therefore consistent across the development
split rather than being driven by a single favorable fold.

This result shows that both components matter:

1. the engineered statistical, temporal, and frequency features provide a
   strong representation;
2. sequential boosting uses the nonlinear structure in those features more
   effectively than either a linear model or the bagged Random Forest.

Histogram gradient boosting passes the predeclared replacement gate and becomes
the current development leader. Detailed activity- and subject-level error
analysis is deferred until the planned feature-based MLP has been evaluated, so
that diagnostics focus on the final development winner.

The reserved test set remains untouched.